# Day 36 · Agent 评测

**配套讲义**: [`days/day-36.md`](../days/day-36.md) ｜ **本地可跑，不需要 GPU**

建 Agent 评测体系：任务完成率 / 工具选择准确率 / 平均轮数 / P95 延迟 / 单次成本；出 `reports/agent_eval_v1.md`，并定位失败集中在哪类任务。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w6.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys
print("python:", sys.version.split()[0])
for m in ("numpy", "PIL", "yaml", "pandas"):
    try:
        mod = __import__(m)
        print(f"  {m:7s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"  {m:7s} ❌ 缺 → pip install {m}")
print("\n→ 本机没 GPU 不影响今天：今天只用纯 Python / numpy")

## 1. 先验证评测逻辑本身

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.eval.agent_eval", "--self-test"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout or r.stderr)

## 2. 设计你自己的任务（这是今天的核心工作）

每条任务要写清「**怎么算通过**」。

In [ ]:
import sys; sys.path.insert(0, "..")
from src.eval.agent_eval import DEFAULT_TASKS, AgentTask

print(f"内置任务 {len(DEFAULT_TASKS)} 条，看两条结构：")
for t in DEFAULT_TASKS[:2]:
    print("=" * 60)
    for k, v in vars(t).items():
        print(f"  {k:22s} {v}")

my_tasks = [
    # AgentTask(query="我要把 A1 退了，尺码不对",
    #           expect_tools=["start_return"],
    #           answer_contains=["退货", "已"], state_key="return_created"),
]
print(f"\n我还需要补 {max(0, 30 - len(DEFAULT_TASKS) - len(my_tasks))} 条")

## 3. 成本核算方法（W8 会用到）

In [ ]:
TOKEN_PRICE = {"in": 0.4 / 1e6, "out": 1.2 / 1e6}   # ¥ / token，按你实际用的算

def session_cost(n_steps, in_tokens, out_tokens, tool_calls):
    llm = in_tokens * TOKEN_PRICE["in"] + out_tokens * TOKEN_PRICE["out"]
    tools = tool_calls * 0.0001      # 假设每次工具调用含内部查询成本
    return llm + tools

c = session_cost(n_steps=2, in_tokens=3200, out_tokens=180, tool_calls=2)
print(f"单次会话成本 ≈ ¥{c:.4f}")
print(f"1000 次会话   ≈ ¥{c*1000:.2f}")
print("\n→ 对照你的定价：¥29/月的店，每月能承受几次会话？")

## 验收清单

- [ ] 任务集 ≥30 条，覆盖查询 / 写操作 / 超范围 / 多轮四类
- [ ] `--self-test` 通过（**评测逻辑本身必须先被验证**）
- [ ] 报告含五个指标：成功率 / 工具准确率 / 平均轮数 / P95 / 单次成本
- [ ] 能定位失败集中在哪一类任务，并给出具体改法
- [ ] `progress/weekly-review.md` 的 W6 段已写；进度表 W6 六天 `[x]`，M6 打卡

**卡住了？** 回看 [`days/day-36.md`](../days/day-36.md) 第五节「容易踩的坑」。

> **明天**：`days/day-37.md` —— W7 Shopify 周：Admin GraphQL